# Peoplesoft Analytics: Medallion Architecture Demo

## Overview

This notebook demonstrates a complete **Medallion Architecture** implementation in Oracle AI Data Platform (AIDP) Workbench using a Peoplesoft analytics use case for the Healthcare industry. The medallion architecture organizes data into three layers:

- **Bronze Layer**: Raw ingested data, stored as-is from source systems
- **Silver Layer**: Cleaned, standardized, and enriched data
- **Gold Layer**: Business-ready aggregates and ML-ready datasets

**Key features**
1. Delta Liquid Clustering throughout all layers to optimize query performance automatically.
2. Use Case: Purchase Order Header Analytics with Medallion Architecture
3. We'll process PO Header records through the complete medallion pipeline, culminating in machine learning models and PO Header Analytics.

**AIDP Environment Setup**
1. This notebook leverages the existing Spark session in your AIDP environment.

## Step 1: Create Peoplesoft for Healthcare Catalog and Schemas

#### Medallion Schema Design

- **bronze**: Raw data landing zone
- **silver**: Cleaned and standardized data
- **gold**: Business analytics and ML-ready data

In [3]:
# Create healthcare catalog and medallion schemas

spark.sql("CREATE CATALOG IF NOT EXISTS peoplesoft")

spark.sql("CREATE SCHEMA IF NOT EXISTS peoplesoft.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS peoplesoft.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS peoplesoft.gold")

print("Healthcare peoplesoft catalog and medallion schemas (bronze, silver, gold) created successfully!")

Healthcare peoplesoft catalog and medallion schemas (bronze, silver, gold) created successfully!


## Bronze Layer: Raw Data Ingestion

### Bronze Layer Purpose

The bronze layer serves as the raw data landing zone where data is ingested from source systems **as-is**, without any transformations. This preserves data integrity and enables:

- **Data lineage**: Complete audit trail from source to consumption
- **Reprocessing**: Ability to reprocess data if business rules change
- **Compliance**: Raw data retention for regulatory requirements

### Clustering Strategy

Cluster by `ingestion_timestamp` , `business_unit`and `po_id` to optimize for:
- Time-based data processing and incremental loads
- po-centric queries during data validation

In [53]:
# Create Bronze Layer Delta table with liquid clustering

#spark.sql("DROP TABLE IF EXISTS healthcare.bronze.po_hdr_raw")
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.bronze.po_hdr_raw (
    business_unit 		STRING,
    po_id 				STRING,  
    po_date 			STRING, -- Raw date string, various formats possible
    vendor_setid 		STRING,
    vendor_id 			STRING,
    status 				STRING,
    currency_code 		STRING,
    total_amount 		STRING,
    ingestion_timestamp TIMESTAMP  -- When data was ingested
)
USING DELTA
CLUSTER BY (ingestion_timestamp, po_id)
""")

print("Bronze layer table created successfully!")
print("Clustering will optimize for time-based ingestion and patient-centric queries.")

Bronze layer table created successfully!
Clustering will optimize for time-based ingestion and patient-centric queries.


### Generate and Ingest Raw Healthcare Data

#### Raw Data Characteristics

Raw data may contain:
- **Inconsistent formatting**: Different date formats, case variations
- **Missing values**: Null or empty fields
- **Data quality issues**: Typos, duplicates, invalid codes
- **Multiple sources**: Different systems with varying schemas

#### Data Generation Strategy

We'll simulate realistic raw healthcare data with some quality issues that would typically be found in bronze layer data.

In [14]:
# Generate realistic raw healthcare diagnosis data with quality issues

import random
from datetime import datetime, timedelta

# Define healthcare data with some inconsistencies (bronze layer characteristics)
PO_HDR_RAW = [
    ("US001",  "SHARE","O","USD","1465.92"),
    ("US001",  "SHARE","O","USD","465.92"),
    ("US001",  "SHARE","S","USD","165.92"),
    ("US001",  "SHARE","S","USD","146.92"),
	("US002",  "SHARE","O","USD","1465.92"),
    ("us002",  "SHARE","S","USD","465.82"),# Case inconsistency
    ("US003",  "SHARE","O","USD","165.93"),
    ("US004",  "share","S","usd","146.42"),# Case inconsistency
    ("us003",   "","", "",""),  # Some missing values
    ("us003",   "","", "",""),  # Some missing values
    ("US003",  "SHARE","O","USD","1465.92"),
    ("us003",  "SHARE","S","USD","465.82"),# Case inconsistency
    ("US004",  "SHARE","O","USD","165.93"),
    ("US004",  "share","S","usd","146.42"),# Case inconsistency("e11.9", "type 2 diabetes mellitus without complications", "medium"),  # Case inconsistency
    ("INVALID", "unk", "O","CUR",""),  # Invalid data
]

VENDOR_ID_RAW = ["V0002", "v0003", "v0004", "V0005", "V0006", ""]  # Case and missing
BUSINESS_UNIT_RAW = ["us002", "US001", "us003", "us005", "US006", "US003", "", "UNKNOWN"] # Case , invalid and missing

# Different date formats to simulate raw data
DATE_FORMATS = ["%Y-%m-%d", "%m/%d/%Y", "%d-%b-%Y", "%Y/%m/%d"]

# Generate raw patient diagnosis records
raw_po_hdr_data = []
base_date = datetime(2024, 1, 1)
ingestion_time = datetime.now()

# Create 1,000 patients with 2-8 diagnoses each, including some data quality issues
for po_num in range(1, 1001):
    po_id = f"P{po_num:07d}"
    
    # Each patient gets 2-8 diagnoses over 12 months
    num_vendor_ids = random.randint(2, 8)
    
    for i in range(num_vendor_ids):
        # Spread diagnoses over 12 months
        days_offset = random.randint(0, 365)
        po_date_obj = base_date + timedelta(days=days_offset)
        
        # Random date format to simulate raw data inconsistency
        date_format = random.choice(DATE_FORMATS)
        po_date_str = po_date_obj.strftime(date_format)
        
        # Select random diagnosis (including some with quality issues)
        business_unit, vendor_setid,status,currency_cd,total_amount = random.choice(PO_HDR_RAW)
        
        # Select random facility and physician (including inconsistencies)
        vendor_id = random.choice(VENDOR_ID_RAW)
        business_unit = random.choice(BUSINESS_UNIT_RAW)
        
        # Occasionally introduce missing values (bronze layer realism)
        if random.random() < 0.05:  # 5% chance of missing data
            vendor_id = None if random.random() < 0.5 else vendor_id
            total_amount = None if random.random() < 0.5 else total_amount
        
        raw_po_hdr_data.append({
            "business_unit": business_unit,
            "po_id": po_id,
            "po_date": po_date_str,
            "vendor_setid": vendor_setid,
            "vendor_id": vendor_id,
            "status": status,
            "currency_code": currency_cd,
            "total_amount": total_amount,
            "ingestion_timestamp": ingestion_time
        })

print(f"Generated {len(raw_po_hdr_data)} raw po header records")
print("Raw data includes formatting inconsistencies, missing values, and data quality issues")
print("Sample raw record:", raw_po_hdr_data[0])

Generated 5050 raw po header records
Raw data includes formatting inconsistencies, missing values, and data quality issues
Sample raw record: {"business_unit": "us003", "po_id": "P0000001", "po_date": "2024-12-16", "vendor_setid": "share", "vendor_id": None, "status": "S", "currency_code": "usd", "total_amount": None, "ingestion_timestamp": datetime.datetime(2026, 5, 11, 16, 41, 47, 767199)}


In [15]:
# Insert raw data into Bronze layer

# Create DataFrame from raw generated data
df_bronze = spark.createDataFrame(raw_po_hdr_data)

# Display schema and sample data
print("Bronze Layer DataFrame Schema:")
df_bronze.printSchema()

print("\nSample Raw Data:")
df_bronze.show(5)

# Insert data into Bronze Delta table
df_bronze.write.mode("overwrite").saveAsTable("healthcare.bronze.po_hdr_raw")

print(f"\nSuccessfully inserted {df_bronze.count()} raw records into healthcare.bronze.po_cust_tbl_raw")
print("Bronze layer preserves raw data as-is for auditability and reprocessing.")

Bronze Layer DataFrame Schema:
root
 |-- business_unit: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- po_date: string (nullable = true)
 |-- po_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- vendor_setid: string (nullable = true)


Sample Raw Data:
+-------------+-------------+--------------------+-----------+--------+------+------------+---------+------------+
|business_unit|currency_code| ingestion_timestamp|    po_date|   po_id|status|total_amount|vendor_id|vendor_setid|
+-------------+-------------+--------------------+-----------+--------+------+------------+---------+------------+
|        us003|          usd|2026-05-11 16:41:...| 2024-12-16|P0000001|     S|        NULL|     NULL|       share|
|        US001|          usd|2026-05-11 16:41:...|11-May-2024|P0000001|     S|      146.42|       


Successfully inserted 5050 raw records into healthcare.bronze.po_cust_tbl_raw
Bronze layer preserves raw data as-is for auditability and reprocessing.


## Silver Layer: Data Cleaning and Standardization

### Silver Layer Purpose

The silver layer transforms raw bronze data into clean, standardized, and enriched datasets suitable for analytics:

- **Data Quality**: Cleansing, standardization, and validation
- **Normalization**: Consistent formats, units, and naming conventions
- **Enrichment**: Adding derived fields, lookups, and business rules
- **Deduplication**: Removing duplicates and handling conflicts

### Table Design: silver.patient_diagnoses_clean

Our silver table includes cleaned and enriched fields:

- **po_id**: Standardized po identifier
- **po_date**: Properly formatted DATE type
- **vendor_setid**: Validated and standardized vendor_setid codes
- **vendor_id**: Cleaned and standardized vendor codes
- **status**: Standardized po status categories
- **currency_code**: Validated currency code
- **total_amount**: total amount
- **is_valid_record**: Data quality flag
- **processing_timestamp**: When record was processed

### Clustering Strategy

Cluster by `po_id` and `po_date` for optimal query performance on:
- PO journey analysis
- Time-based peoplesoft analytics
- Validity performance metrics

In [67]:
# Create Silver Layer Delta table with liquid clustering

spark.sql("DROP TABLE IF EXISTS healthcare.silver.po_hdr_clean")
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.silver.po_hdr_clean (
    business_unit 		STRING,
    po_id 				STRING,  
    po_date 			DATE, -- Raw date string, various formats possible
    vendor_setid 		STRING,
    vendor_id 			STRING,
    status 				STRING,
    currency_code 		STRING,
    total_amount 		DECIMAL(18,2),
    is_valid_record 	BOOLEAN,
    data_quality_score 	DOUBLE,
    processing_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY (po_id, business_unit)
""")

print("Silver layer table created successfully!")
print("Clustering optimizes for po-header-centric and time-based analytics.")

Silver layer table created successfully!
Clustering optimizes for po-header-centric and time-based analytics.


In [71]:
# Transform bronze data to silver layer with cleaning and standardization

from pyspark.sql.functions import *
from pyspark.sql.types import DateType

# Read bronze data
bronze_df = spark.table("healthcare.bronze.po_hdr_raw")

# Data cleaning and standardization transformations
silver_df = bronze_df.withColumn(
    "po_date",
    coalesce(
        to_date("po_date", "yyyy-MM-dd"),
        to_date("po_date", "MM/dd/yyyy"),
        to_date("po_date", "dd-MMM-yyyy"),
        to_date("po_date", "yyyy/MM/dd")
    )
).withColumn(
    "currency_code",
    upper(trim("currency_code"))
).withColumn(
    "business_unit",
    when(
        upper(col("business_unit")).isin("US001", "US002", "US003", "US004", "US005", "US006"),
        upper(col("business_unit"))
    ).otherwise("UNKNOWN")
).withColumn(
    "status",
    when(upper(trim("status")).isin(["O", "S", "M", "L"]), 
         initcap(trim("status")))
    .otherwise("U")
).withColumn(
    "vendor_setid",
    when(trim("vendor_setid") != "", upper(trim("vendor_setid")))
    .otherwise("UNKNOWN")
).withColumn(
    "vendor_id",
    when(trim("vendor_id") != "", upper(trim("vendor_id")))
    .otherwise("UNKNOWN")
).withColumn(
    "total_amount",
    col("total_amount").cast("decimal(18,2)")
).withColumn(
    "is_valid_record",
    (col("business_unit").isNotNull() & upper(col("business_unit")).isin("US001", "US002", "US003", "US004", "US005", "US006")) &     
    (col("po_id").isNotNull()) & 
    (col("po_date").isNotNull()) & 
    (col("vendor_setid").isNotNull() & upper(col("vendor_setid")).isin("SHARE")) &
    (col("vendor_id").isNotNull() & upper(col("vendor_id")).isin("V0002", "v0003", "v0004", "V0005", "V0006")) &
    (col("status").isNotNull() & upper(col("status")).isin(["O", "S", "M", "L"])) &
    (col("currency_code").isNotNull() & upper(col("currency_code")).isin('USD','GBP')) &
    (col("total_amount").isNotNull()) &
    (length(trim("vendor_id")) > 0)
).withColumn(
    "data_quality_score",
    (when(col("po_id").isNotNull(), 0.2).otherwise(0) +
     when(col("po_date").isNotNull(), 0.2).otherwise(0) +
     when(col("business_unit").isNotNull() & (length(trim("business_unit")) > 0), 0.2).otherwise(0) +
     when(col("currency_code") != "Unknown", 0.2).otherwise(0) +
     when(col("status") != "UNKNOWN", 0.2).otherwise(0))
).withColumn(
    "processing_timestamp",
    current_timestamp()
).filter(
    col("is_valid_record") == True  # Only keep valid records in silver layer
)

# Remove duplicates based on patient_id, diagnosis_date, diagnosis_code
silver_df = silver_df.dropDuplicates(["po_id", "business_unit"])
silver_df = silver_df.drop("ingestion_timestamp")

# Insert cleaned data into silver layer
silver_df.write.mode("overwrite").saveAsTable("healthcare.silver.po_hdr_clean")

print(f"Successfully processed {silver_df.count()} clean records into healthcare.silver.po_hdr_clean")
print("Silver layer provides standardized, validated, and enriched data for analytics.")

Successfully processed 1274 clean records into healthcare.silver.po_hdr_clean
Silver layer provides standardized, validated, and enriched data for analytics.


In [72]:
# Validate silver layer data quality improvements

print("=== Silver Layer Data Quality Validation ===")

# Compare bronze vs silver data quality
bronze_count = spark.table("healthcare.bronze.po_hdr_raw").count()
silver_count = spark.table("healthcare.silver.po_hdr_clean").count()

print(f"Bronze layer records: {bronze_count}")
print(f"Silver layer records: {silver_count}")
print(f"Data quality improvement: {((silver_count/bronze_count)*100):.1f}% valid records retained")

# Show data quality distribution
quality_distribution = spark.table("healthcare.silver.po_hdr_clean").groupBy("data_quality_score").count().orderBy("data_quality_score")
quality_distribution.show()

# Sample cleaned records
print("\nSample Cleaned Records:")
spark.table("healthcare.silver.po_hdr_clean").show(5)

=== Silver Layer Data Quality Validation ===


Bronze layer records: 5050
Silver layer records: 1274
Data quality improvement: 25.2% valid records retained


+------------------+-----+
|data_quality_score|count|
+------------------+-----+
|               1.0| 1274|
+------------------+-----+


Sample Cleaned Records:


+-------------+--------+----------+------------+---------+------+-------------+------------+---------------+------------------+--------------------+
|business_unit|   po_id|   po_date|vendor_setid|vendor_id|status|currency_code|total_amount|is_valid_record|data_quality_score|processing_timestamp|
+-------------+--------+----------+------------+---------+------+-------------+------------+---------------+------------------+--------------------+
|        US003|P0000002|2024-02-09|       SHARE|    V0002|     S|          USD|      146.42|           true|               1.0|2026-05-12 01:09:...|
|        US003|P0000003|2024-09-26|       SHARE|    V0006|     O|          USD|      165.93|           true|               1.0|2026-05-12 01:09:...|
|        US006|P0000003|2024-01-05|       SHARE|    V0002|     O|          USD|     1465.92|           true|               1.0|2026-05-12 01:09:...|
|        US006|P0000004|2024-10-07|       SHARE|    V0006|     S|          USD|      146.42|           tru

## Gold Layer: Business Analytics and ML-Ready Data

### Gold Layer Purpose

The gold layer contains business-ready datasets optimized for:

- **Analytics Dashboards**: Aggregated metrics and KPIs
- **Reporting**: Standardized business views
- **Machine Learning**: Feature-engineered datasets
- **Downstream Applications**: Clean, fast-access data

### Gold Layer Tables

We'll create multiple gold tables:

1. **gold.po_summary_daily**: PO-level aggregates
2. **gold.po_status_analytics**: PO Status analytics
3. **gold.po_vendor_performance**: PO and vendor metrics
4. **gold.vendor_summary_features**: ML-ready features for vendor analytics

### Clustering Strategies

- PO summary: Cluster by `po_id`
- PO analytics: Cluster by `po_id`, `month`
- Vendor performance: Cluster by `vendor_id`, `month`
- ML features: Cluster by `po_id`

In [78]:
# Create Gold Layer tables with liquid clustering

# Daily po summary
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_summary_daily
USING DELTA
AS
SELECT
    po_date,
    business_unit,
    currency_code,
    COUNT(*) AS po_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    MIN(total_amount) AS min_po_amount,
    MAX(total_amount) AS max_po_amount,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    AVG(data_quality_score) AS avg_data_quality_score,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    po_date,
    business_unit,
    currency_code
""")

# Monthly PO summary
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_summary_monthly
USING DELTA
AS
SELECT
    YEAR(po_date) AS po_year,
    MONTH(po_date) AS po_month,
    business_unit,
    currency_code,
    COUNT(*) AS po_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    MIN(total_amount) AS min_po_amount,
    MAX(total_amount) AS max_po_amount,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    AVG(data_quality_score) AS avg_data_quality_score,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    YEAR(po_date),
    MONTH(po_date),
    business_unit,
    currency_code
""")

# po_status_analytics
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_status_analytics
USING DELTA
AS
SELECT
    business_unit,
    status,
    currency_code,
    COUNT(*) AS po_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    AVG(data_quality_score) AS avg_data_quality_score,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    business_unit,
    status,
    currency_code
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_vendor_performance
USING DELTA
AS
SELECT
    vendor_setid,
    vendor_id,
    business_unit,
    currency_code,
    COUNT(*) AS po_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    MIN(po_date) AS first_po_date,
    MAX(po_date) AS last_po_date,
    AVG(data_quality_score) AS avg_data_quality_score,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    vendor_setid,
    vendor_id,
    business_unit,
    currency_code
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_business_unit_performance
USING DELTA
AS
SELECT
    business_unit,
    currency_code,
    COUNT(*) AS po_count,
    COUNT(DISTINCT vendor_id) AS distinct_vendor_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    MIN(total_amount) AS min_po_amount,
    MAX(total_amount) AS max_po_amount,
    AVG(data_quality_score) AS avg_data_quality_score,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    business_unit,
    currency_code
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_data_quality_summary
USING DELTA
AS
SELECT
    business_unit,
    status,
    COUNT(*) AS total_record_count,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    ROUND(100.0 * SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) / COUNT(*), 2) AS valid_record_pct,
    AVG(data_quality_score) AS avg_data_quality_score,
    MIN(data_quality_score) AS min_data_quality_score,
    MAX(data_quality_score) AS max_data_quality_score,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
GROUP BY
    business_unit,
    status
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.gold_po_kpi_dashboard
USING DELTA
AS
SELECT
    CURRENT_DATE() AS snapshot_date,
    COUNT(*) AS total_po_count,
    SUM(total_amount) AS total_po_amount,
    AVG(total_amount) AS avg_po_amount,
    COUNT(DISTINCT business_unit) AS business_unit_count,
    COUNT(DISTINCT vendor_id) AS vendor_count,
    SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
    SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
    ROUND(100.0 * SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) / COUNT(*), 2) AS valid_record_pct,
    AVG(data_quality_score) AS avg_data_quality_score,
    MAX(processing_timestamp) AS last_processing_timestamp
FROM healthcare.silver.po_hdr_clean
""")

print("Gold layer tables created successfully!")
print("Each table is optimized with liquid clustering for specific query patterns.")

Gold layer tables created successfully!
Each table is optimized with liquid clustering for specific query patterns.


In [82]:
silver_df = spark.table("healthcare.silver.po_hdr_clean")
silver_df.createOrReplaceTempView("silver_po_hdr")

In [83]:
# Check the row counts for these tables 

#spark.sql("select count(*) from healthcare.gold.gold_po_summary_daily").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_summary_monthly").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_status_analytics").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_vendor_performance").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_business_unit_performance").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_data_quality_summary").show()
#spark.sql("select count(*) from healthcare.gold.gold_po_kpi_dashboard").show()
vendor_features_sql = """
SELECT
    vendor_setid,
    vendor_id,
    business_unit,
    currency_code,
    total_pos,
    CAST(total_po_amount AS DECIMAL(18,2)) AS total_po_amount,
    ROUND(avg_po_amount, 2) AS avg_po_amount,
    CAST(min_po_amount AS DECIMAL(18,2)) AS min_po_amount,
    CAST(max_po_amount AS DECIMAL(18,2)) AS max_po_amount,
    active_days,
    active_months,
    first_po_date,
    last_po_date,
    days_since_last_po,
    valid_record_count,
    invalid_record_count,
    ROUND(valid_record_pct, 2) AS valid_record_pct,
    ROUND(avg_data_quality_score, 4) AS avg_data_quality_score,
    CASE WHEN total_po_amount > 50000 THEN 1 ELSE 0 END AS high_spend_vendor_flag,
    CASE WHEN avg_data_quality_score < 0.85 OR invalid_record_count > 0 THEN 1 ELSE 0 END AS low_quality_vendor_flag,
    CASE
        WHEN total_po_amount > 50000
          OR avg_data_quality_score < 0.85
          OR invalid_record_count > 0
        THEN 1 ELSE 0
    END AS vendor_risk_label,
    CURRENT_TIMESTAMP() AS feature_timestamp
FROM (
    SELECT
        vendor_setid,
        vendor_id,
        business_unit,
        currency_code,
        COUNT(*) AS total_pos,
        SUM(total_amount) AS total_po_amount,
        AVG(total_amount) AS avg_po_amount,
        MIN(total_amount) AS min_po_amount,
        MAX(total_amount) AS max_po_amount,
        COUNT(DISTINCT po_date) AS active_days,
        COUNT(DISTINCT DATE_FORMAT(po_date, 'yyyy-MM')) AS active_months,
        MIN(po_date) AS first_po_date,
        MAX(po_date) AS last_po_date,
        DATEDIFF(CURRENT_DATE(), MAX(po_date)) AS days_since_last_po,
        SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
        SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
        100.0 * SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) / COUNT(*) AS valid_record_pct,
        AVG(data_quality_score) AS avg_data_quality_score
    FROM silver_po_hdr
    GROUP BY
        vendor_setid,
        vendor_id,
        business_unit,
        currency_code
) x
"""
#spark.sql(vendor_features_sql).show()

+------------+---------+-------------+-------------+---------+---------------+-------------+-------------+-------------+-----------+-------------+-------------+------------+------------------+------------------+--------------------+----------------+----------------------+----------------------+-----------------------+-----------------+--------------------+
|vendor_setid|vendor_id|business_unit|currency_code|total_pos|total_po_amount|avg_po_amount|min_po_amount|max_po_amount|active_days|active_months|first_po_date|last_po_date|days_since_last_po|valid_record_count|invalid_record_count|valid_record_pct|avg_data_quality_score|high_spend_vendor_flag|low_quality_vendor_flag|vendor_risk_label|   feature_timestamp|
+------------+---------+-------------+-------------+---------+---------------+-------------+-------------+-------------+-----------+-------------+-------------+------------+------------------+------------------+--------------------+----------------+----------------------+----------

In [85]:
vendor_features_df = spark.sql(vendor_features_sql)
vendor_features_df.createOrReplaceTempView("vendor_summary_features_src")
spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.vendor_summary_features 
USING DELTA 
AS
SELECT * FROM vendor_summary_features_src
""")

In [91]:
bu_features_sql = """
SELECT
    business_unit,
    currency_code,
    total_pos,
    distinct_vendors,
    CAST(total_po_amount AS DECIMAL(18,2)) AS total_po_amount,
    ROUND(avg_po_amount, 2) AS avg_po_amount,
    CAST(min_po_amount AS DECIMAL(18,2)) AS min_po_amount,
    CAST(max_po_amount AS DECIMAL(18,2)) AS max_po_amount,
    active_days,
    active_months,
    first_po_date,
    last_po_date,
    days_since_last_po,
    valid_record_count,
    invalid_record_count,
    ROUND(valid_record_pct, 2) AS valid_record_pct,
    ROUND(avg_data_quality_score, 4) AS avg_data_quality_score,
    CASE WHEN total_po_amount > 100000 THEN 1 ELSE 0 END AS high_spend_bu_flag,
    CASE WHEN distinct_vendors > 10 THEN 1 ELSE 0 END AS high_vendor_diversity_flag,
    CASE WHEN avg_data_quality_score < 0.85 OR invalid_record_count > 0 THEN 1 ELSE 0 END AS low_quality_bu_flag,
    CASE
        WHEN total_po_amount > 100000
          OR distinct_vendors > 10
          OR avg_data_quality_score < 0.85
          OR invalid_record_count > 0
        THEN 1 ELSE 0
    END AS procurement_risk_label,
    CURRENT_TIMESTAMP() AS feature_timestamp
FROM (
    SELECT
        business_unit,
        currency_code,
        COUNT(*) AS total_pos,
        COUNT(DISTINCT vendor_id) AS distinct_vendors,
        SUM(total_amount) AS total_po_amount,
        AVG(total_amount) AS avg_po_amount,
        MIN(total_amount) AS min_po_amount,
        MAX(total_amount) AS max_po_amount,
        COUNT(DISTINCT po_date) AS active_days,
        COUNT(DISTINCT DATE_FORMAT(po_date, 'yyyy-MM')) AS active_months,
        MIN(po_date) AS first_po_date,
        MAX(po_date) AS last_po_date,
        DATEDIFF(CURRENT_DATE(), MAX(po_date)) AS days_since_last_po,
        SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
        SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
        100.0 * SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) / COUNT(*) AS valid_record_pct,
        AVG(data_quality_score) AS avg_data_quality_score
    FROM silver_po_hdr
    GROUP BY
        business_unit,
        currency_code
) x
"""

In [93]:
bu_features_df = spark.sql(bu_features_sql)
bu_features_df.createOrReplaceTempView("business_unit_procurement_features_src")

spark.sql("""
CREATE TABLE if not exists healthcare.gold.business_unit_procurement_features
SELECT * FROM business_unit_procurement_features_src
""")

DataFrame[]

In [94]:
po_anomaly_sql = """
SELECT
    s.po_id,
    s.business_unit,
    s.vendor_setid,
    s.vendor_id,
    s.po_date,
    DATE_FORMAT(s.po_date, 'yyyy-MM') AS po_month,
    s.status,
    s.currency_code,
    CAST(s.total_amount AS DECIMAL(18,2)) AS total_amount,
    s.is_valid_record,
    s.data_quality_score,
    v.vendor_po_count,
    ROUND(v.vendor_avg_po_amount, 2) AS vendor_avg_po_amount,
    ROUND(v.vendor_max_po_amount, 2) AS vendor_max_po_amount,
    ROUND(b.bu_avg_po_amount, 2) AS bu_avg_po_amount,
    ROUND(b.bu_max_po_amount, 2) AS bu_max_po_amount,
    ROUND(CASE WHEN v.vendor_avg_po_amount = 0 THEN NULL ELSE s.total_amount / v.vendor_avg_po_amount END, 4) AS amount_vs_vendor_avg_ratio,
    ROUND(CASE WHEN b.bu_avg_po_amount = 0 THEN NULL ELSE s.total_amount / b.bu_avg_po_amount END, 4) AS amount_vs_bu_avg_ratio,
    CASE WHEN s.vendor_id IS NULL OR TRIM(s.vendor_id) = '' THEN 1 ELSE 0 END AS invalid_vendor_flag,
    CASE WHEN s.is_valid_record = FALSE OR s.data_quality_score < 0.85 THEN 1 ELSE 0 END AS low_quality_flag,
    CASE
        WHEN v.vendor_avg_po_amount > 0 AND s.total_amount > (2.0 * v.vendor_avg_po_amount) THEN 1
        WHEN b.bu_avg_po_amount > 0 AND s.total_amount > (2.0 * b.bu_avg_po_amount) THEN 1
        ELSE 0
    END AS high_amount_outlier_flag,
    CASE
        WHEN s.is_valid_record = FALSE
          OR s.data_quality_score < 0.85
          OR (v.vendor_avg_po_amount > 0 AND s.total_amount > (2.0 * v.vendor_avg_po_amount))
          OR (b.bu_avg_po_amount > 0 AND s.total_amount > (2.0 * b.bu_avg_po_amount))
        THEN 1 ELSE 0
    END AS po_anomaly_label,
    CURRENT_TIMESTAMP() AS feature_timestamp
FROM silver_po_hdr s
LEFT JOIN (
    SELECT
        vendor_setid,
        vendor_id,
        business_unit,
        COUNT(*) AS vendor_po_count,
        AVG(total_amount) AS vendor_avg_po_amount,
        MAX(total_amount) AS vendor_max_po_amount
    FROM silver_po_hdr
    GROUP BY vendor_setid, vendor_id, business_unit
) v
    ON s.vendor_setid = v.vendor_setid
   AND s.vendor_id = v.vendor_id
   AND s.business_unit = v.business_unit
LEFT JOIN (
    SELECT
        business_unit,
        AVG(total_amount) AS bu_avg_po_amount,
        MAX(total_amount) AS bu_max_po_amount
    FROM silver_po_hdr
    GROUP BY business_unit
) b
    ON s.business_unit = b.business_unit
"""

In [95]:
po_anomaly_df = spark.sql(po_anomaly_sql)
po_anomaly_df.createOrReplaceTempView("po_header_anomaly_features_src")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.po_header_anomaly_features
SELECT * FROM po_header_anomaly_features_src
""")

DataFrame[]

In [96]:
vendor_monthly_sql = """
SELECT
    vendor_id,
    business_unit,
    po_month,
    total_pos,
    CAST(total_po_amount AS DECIMAL(18,2)) AS total_po_amount,
    ROUND(avg_po_amount, 2) AS avg_po_amount,
    valid_record_count,
    invalid_record_count,
    ROUND(avg_data_quality_score, 4) AS avg_data_quality_score,
    ROUND(
        CASE
            WHEN prev_total_po_amount IS NULL OR prev_total_po_amount = 0 THEN NULL
            ELSE ((total_po_amount - prev_total_po_amount) / prev_total_po_amount) * 100
        END, 2
    ) AS month_over_month_amount_change_pct,
    ROUND(
        CASE
            WHEN prev_total_pos IS NULL OR prev_total_pos = 0 THEN NULL
            ELSE ((total_pos - prev_total_pos) / prev_total_pos) * 100
        END, 2
    ) AS month_over_month_po_change_pct,
    CURRENT_TIMESTAMP() AS feature_timestamp
FROM (
    SELECT
        vendor_id,
        business_unit,
        po_month,
        total_pos,
        total_po_amount,
        avg_po_amount,
        valid_record_count,
        invalid_record_count,
        avg_data_quality_score,
        LAG(total_po_amount) OVER (PARTITION BY vendor_id, business_unit ORDER BY po_month) AS prev_total_po_amount,
        LAG(total_pos) OVER (PARTITION BY vendor_id, business_unit ORDER BY po_month) AS prev_total_pos
    FROM (
        SELECT
            vendor_id,
            business_unit,
            DATE_FORMAT(po_date, 'yyyy-MM') AS po_month,
            COUNT(*) AS total_pos,
            SUM(total_amount) AS total_po_amount,
            AVG(total_amount) AS avg_po_amount,
            SUM(CASE WHEN is_valid_record THEN 1 ELSE 0 END) AS valid_record_count,
            SUM(CASE WHEN NOT is_valid_record THEN 1 ELSE 0 END) AS invalid_record_count,
            AVG(data_quality_score) AS avg_data_quality_score
        FROM silver_po_hdr
        GROUP BY
            vendor_id,
            business_unit,
            DATE_FORMAT(po_date, 'yyyy-MM')
    ) m
) x
"""

In [97]:
vendor_monthly_df = spark.sql(vendor_monthly_sql)
vendor_monthly_df.createOrReplaceTempView("vendor_monthly_trend_features_src")

spark.sql("""
CREATE TABLE IF NOT EXISTS healthcare.gold.vendor_monthly_trend_features
SELECT * FROM vendor_monthly_trend_features_src
""")

DataFrame[]

In [99]:
ml_df = spark.sql("""
SELECT
    CAST(vendor_po_count AS DOUBLE) AS vendor_po_count,
    vendor_avg_po_amount,
    vendor_max_po_amount,
    bu_avg_po_amount,
    bu_max_po_amount,
    amount_vs_vendor_avg_ratio,
    amount_vs_bu_avg_ratio,
    CAST(invalid_vendor_flag AS INT) AS invalid_vendor_flag,
    CAST(low_quality_flag AS INT) AS low_quality_flag,
    CAST(high_amount_outlier_flag AS INT) AS high_amount_outlier_flag,
    CAST(po_anomaly_label AS INT) AS label
FROM healthcare.gold.po_header_anomaly_features
WHERE total_amount IS NOT NULL
""")

ml_df.show(10, truncate=False)

+---------------+--------------------+--------------------+----------------+----------------+--------------------------+----------------------+-------------------+----------------+------------------------+-----+
|vendor_po_count|vendor_avg_po_amount|vendor_max_po_amount|bu_avg_po_amount|bu_max_po_amount|amount_vs_vendor_avg_ratio|amount_vs_bu_avg_ratio|invalid_vendor_flag|low_quality_flag|high_amount_outlier_flag|label|
+---------------+--------------------+--------------------+----------------+----------------+--------------------------+----------------------+-------------------+----------------+------------------------+-----+
|129.0          |575.24              |1465.92             |528.34          |1465.92         |0.2545                    |0.2771                |0                  |0               |0                       |0    |
|155.0          |505.28              |1465.92             |528.34          |1465.92         |0.3284                    |0.3141                |0        

## Step 7: Demonstrate Gold Layer Analytics

### Gold Layer Benefits

The gold layer provides optimized access to business-critical insights:

- **Fast Analytics**: Pre-aggregated data for dashboards
- **Consistent Reporting**: Standardized business metrics
- **Predictive Insights**: ML-ready feature sets
- **Operational Intelligence**: Real-time performance monitoring

In [128]:
# Demonstrate Gold Layer analytics capabilities
print("=== Gold Layer Analytics Demonstration ===")
print("=== PO header summary analytics (Gold layer processing) ===")
# Patient summary analytics
full_name="healthcare.gold.gold_po_summary_daily"
df=spark.sql("""
select po_date, business_unit,po_count,total_po_amount,avg_po_amount,min_po_amount,max_po_amount,valid_record_count,invalid_record_count 
from healthcare.gold.gold_po_summary_daily
""")
display(df.limit(20))
print("=== PO header summary analytics (Aggregate based on BU) ===")
df=spark.sql("""
select business_unit,
sum(po_count) as po_count,
sum(total_po_amount) as total_po_amount,
avg(avg_po_amount) as avg_po_amount,
min(min_po_amount) as min_po_amount,
max(max_po_amount) as max_po_amount,
sum(valid_record_count) as total_valid_records,
sum(invalid_record_count) as total_invalid_records
from healthcare.gold.gold_po_summary_daily
group by business_unit
""")
display(df)


=== Gold Layer Analytics Demonstration ===
=== PO header summary analytics (Gold layer processing) ===


=== PO header summary analytics (Aggregate based on BU) ===


In [140]:
# Vendor performance analytics
print("\nVendor Performance Analytics:")
df=spark.sql("select business_unit,vendor_id,po_count,total_po_amount, cast(avg_po_amount as decimal(10,2)) as avg_po_amount,first_po_date,last_po_date from healthcare.gold.gold_po_vendor_performance")
display(df)
df=spark.sql("select vendor_id,sum(po_count),sum(total_po_amount), cast(avg(avg_po_amount) as decimal(10,2)) as avg_po_amount,min(first_po_date),max(last_po_date) from healthcare.gold.gold_po_vendor_performance group by vendor_id")
display(df)


Vendor Performance Analytics:


In [149]:
# po status analytics
print("\nVendor Performance Analytics:")
df=spark.sql("select business_unit,status,po_count,total_po_amount,cast(avg_po_amount as decimal(10,2)) as avg_po_amount from healthcare.gold.gold_po_status_analytics")
display(df)
df=spark.sql("select status,sum(po_count),sum(total_po_amount), cast(avg(avg_po_amount) as decimal(10,2)) as avg_po_amount,sum(valid_record_count) as valid_record_count from healthcare.gold.gold_po_status_analytics group by status")
display(df)


Vendor Performance Analytics:


In [159]:
# ML - po anomaly features analysis
df=spark.sql("select * from healthcare.gold.po_header_anomaly_features")
display(df.limit(5))
df=spark.sql("select po_anomaly_label, count(*) as record_count from healthcare.gold.po_header_anomaly_features group by po_anomaly_label")
display(df)


In [164]:
# vendor monthly trend features
df=spark.sql("select vendor_id,business_unit, po_month, sum(total_pos),sum(total_po_amount) as total_pos from healthcare.gold.vendor_monthly_trend_features group by business_unit, vendor_id, po_month")
display(df.limit(5))

In [167]:
df=spark.sql("select * from healthcare.gold.gold_po_data_quality_summary")
display(df)